# SI Figures S16-S18: predicted vs experimental solvent-induced shifts by reference solvent

Per solvent, implicit (PCM) and explicit (Desmond) predicted solvent-induced shift differences vs
measured (y=x ideal), for three reference solvents: **S16** chloroform, **S17** benzene, **S18**
solvent-averaged. One panel per solvent, each point a proton site.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import matplotlib.pyplot as plt

import delta22
import delta22_plots
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
# Figure 4 uses one DFT method for the proton sites; differences are taken against a reference
# solvent, and the solvent_mean pseudo-solvent is added for the solvent-averaged reference.
METHOD, BASIS, GEOM = "b3lyp_d3bj", "pcSseg2", "aimnet2"
q = delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False)
one = q[(q["sap_nmr_method"] == METHOD) & (q["sap_basis"] == BASIS) & (q["sap_geometry_type"] == GEOM)]
one = delta22.add_solvent_mean(one)
print(len(one), "rows for", METHOD, BASIS, GEOM)

In [ ]:
FIG4C_LABELS = {
    "chloroform": "CDCl3", "dichloromethane": "DCM", "tetrahydrofuran": "THF",
    "acetonitrile": "MeCN", "dimethylsulfoxide": "DMSO", "acetone": "acetone",
    "methanol": "MeOH", "TIP4P": "TIP4P", "trifluoroethanol": "TFE",
    "benzene": "benzene", "toluene": "toluene", "chlorobenzene": "chlorobenzene",
}

In [ ]:
# panel order matches the canonical figures; a different order from delta22.DESMOND_SOLVENTS
SI_S16_18_SOLVENT_ORDER = [
    "chloroform", "tetrahydrofuran", "dichloromethane", "acetone", "acetonitrile",
    "dimethylsulfoxide", "trifluoroethanol", "methanol", "TIP4P", "benzene", "toluene", "chlorobenzene",
]

# panel titles use the full solvent name; axis labels use the compact NMR abbreviations (THF, DCM,
# DMSO, TFE, CDCl3); the reference is CDCl3 / Benzene / "Avg Corr." for S16 / S17 / S18.
TITLE_LABEL = {"chloroform": "Chloroform", "tetrahydrofuran": "Tetrahydrofuran",
               "dichloromethane": "Dichloromethane", "acetone": "Acetone", "acetonitrile": "Acetonitrile",
               "dimethylsulfoxide": "Dimethylsulfoxide", "trifluoroethanol": "Trifluoroethanol",
               "methanol": "Methanol", "TIP4P": "TIP4P", "benzene": "Benzene", "toluene": "Toluene",
               "chlorobenzene": "Chlorobenzene"}
AXIS_LABEL = {**TITLE_LABEL, "chloroform": "CDCl3", "tetrahydrofuran": "THF",
              "dichloromethane": "DCM", "dimethylsulfoxide": "DMSO", "trifluoroethanol": "TFE"}
REF_LABEL = {"chloroform": "CDCl3", "benzene": "Benzene", "solvent_mean": "Avg Corr."}

In [ ]:
for figure, reference in [("S16", "chloroform"), ("S17", "benzene"), ("S18", "solvent_mean")]:
    sh = delta22.solvent_induced_shifts(one, reference, SI_S16_18_SOLVENT_ORDER, nucleus="H", explicit="desmond")
    implicit_fit = delta22.fit_differences_to_experimental(sh, "implicit_diff")
    explicit_fit = delta22.fit_differences_to_experimental(sh, "explicit_diff")
    print(f"{figure}: reference={reference:12s} n={explicit_fit['n']:4d}  "
          f"implicit fit RMSE={implicit_fit['rmse']:.3f}  explicit fit RMSE={explicit_fit['rmse']:.3f}")
    panel_solvents = [s for s in SI_S16_18_SOLVENT_ORDER if s != reference]
    delta22_plots.plot_shift_prediction_scatter_grid(
        sh, panel_solvents, REF_LABEL[reference], AXIS_LABEL, TITLE_LABEL,
        save_path=figure_path(f"si_figure_{figure.lower()}.png"))
    plt.show()